# AED multiclasse completo — áudio, MFCC, log-mel, VAD e TFLite

Este notebook substitui o protótipo binário `perigo / nao_perigo` por um **AED multiclasse** voltado ao projeto de violência embarcada.

Classes:

1. `ambiente_normal`
2. `grito_panico`
3. `choro_distress`
4. `voz_agressiva`
5. `impacto`
6. `vidro_quebrando`
7. `tiro`

Pipeline principal:

```text
áudio mono 16 kHz
→ janelas de 2 segundos
→ log-mel + MFCC
→ CNN leve de duas entradas
→ classe do evento + confiança
→ suavização temporal
→ TFLite
```

Arquitetura paralela do projeto:

```text
áudio bruto
├── AED contínuo: este notebook
├── VAD → Whisper → NLP
└── GCC-PHAT / TDOA / DOA
```

**O VAD não bloqueia o AED.** Tiro, impacto e vidro quebrando podem ocorrer sem fala.

Fontes automáticas usadas:

- **ESC-50**: ambiente, choro, vidro quebrando, batida em porta e negativos difíceis;
- **TUT Rare Sound Events 2017**: choro de bebê, tiro e vidro quebrando;
- **Kaggle**: gritos e pânico, vozes agressivas, impactos e choro de angústia.

O notebook separa os grupos originais antes do augmentation, evitando que versões do mesmo áudio apareçam no treino e no teste.

## Antes de executar

No Colab:

```text
Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU
```

A chave antiga do Freesound apareceu escrita no notebook anterior. Ela deve ser revogada e substituída. Este notebook solicita a chave com `getpass`, sem gravá-la no arquivo.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install                 librosa==0.11.0                 soundfile                 scikit-learn                 pandas                 matplotlib                 seaborn                 requests                 tqdm                 pyyaml                 webrtcvad-wheels \
                kagglehub

In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any
from getpass import getpass
from IPython.display import Audio, display
from tqdm.auto import tqdm

import io
import json
import math
import random
import shutil
import subprocess
import time
import zipfile

import kagglehub
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import soundfile as sf
import tensorflow as tf
import webrtcvad

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

## 1. Configuração geral

In [ ]:
@dataclass(frozen=True)
class ConfiguracaoAED:
    taxa_amostragem: int = 16000
    duracao_janela: float = 2.0
    passo_janela_treino: float = 1.0
    passo_janela_inferencia: float = 0.5

    n_fft: int = 512
    hop_length: int = 256
    n_mels: int = 64
    n_mfcc: int = 40
    fmin: int = 40
    fmax: int = 7600

    max_fontes_por_classe: int = 260
    max_janelas_por_fonte: int = 2
    copias_augmentation_treino: int = 1

    proporcao_treino: float = 0.70
    proporcao_validacao: float = 0.15
    proporcao_teste: float = 0.15

    epocas: int = 45
    tamanho_lote: int = 32
    paciencia: int = 8
    taxa_aprendizado: float = 0.001
    semente: int = 42

    max_freesound_por_classe: int = 140
    max_por_query_freesound: int = 35
    duracao_min_freesound: float = 0.5
    duracao_max_freesound: float = 10.0

    minimo_eventos_consecutivos: int = 2

    usar_esc50: bool = True
    usar_tut_rare: bool = True
    usar_freesound: bool = True


CFG = ConfiguracaoAED()

CLASSES = [
    "ambiente_normal",
    "grito_panico",
    "choro_distress",
    "voz_agressiva",
    "impacto",
    "vidro_quebrando",
    "tiro",
]

ID_CLASSE = {
    nome: indice
    for indice, nome in enumerate(CLASSES)
}

RAIZ = Path("/content/aed_multiclasse")
DADOS = RAIZ / "datasets"
ESC50 = DADOS / "esc50"
TUT = DADOS / "tut_rare"
FREESOUND = DADOS / "freesound"
PROCESSADO = RAIZ / "processado"
MODELOS = RAIZ / "modelos"
RESULTADOS = RAIZ / "resultados"
ENTREGA = RAIZ / "entrega"

for pasta in (
    DADOS,
    ESC50,
    TUT,
    FREESOUND,
    PROCESSADO,
    MODELOS,
    RESULTADOS,
    ENTREGA,
):
    pasta.mkdir(parents=True, exist_ok=True)

random.seed(CFG.semente)
np.random.seed(CFG.semente)
tf.random.set_seed(CFG.semente)

display(pd.DataFrame([asdict(CFG)]))
print("Classes:", ID_CLASSE)

## 2. Baixar ESC-50 e TUT Rare Sound Events

O notebook baixa somente o arquivo de eventos isolados do TUT, não os 17,5 GB completos.

O TUT fornece:

- `baby cry`
- `glass break`
- `gun shot`

O ESC-50 fornece:

- `crying_baby`
- `glass_breaking`
- `door_wood_knock`
- diversas classes normais e negativos difíceis, incluindo `fireworks`, `thunderstorm` e `siren`.

In [ ]:
URL_ESC50 = (
    "https://github.com/karolpiczak/ESC-50/"
    "archive/refs/heads/master.zip"
)

URL_TUT_EVENTOS = (
    "https://zenodo.org/records/401395/files/"
    "TUT-rare-sound-events-2017-development."
    "source_data_events.zip?download=1"
)


def baixar(
    url: str,
    destino: Path,
) -> None:
    if destino.exists() and destino.stat().st_size > 0:
        print("Já existe:", destino)
        return

    with requests.get(
        url,
        stream=True,
        timeout=60,
    ) as resposta:
        resposta.raise_for_status()
        total = int(
            resposta.headers.get(
                "content-length",
                0,
            )
        )

        with destino.open("wb") as arquivo:
            with tqdm(
                total=total,
                unit="B",
                unit_scale=True,
                desc=destino.name,
            ) as barra:
                for bloco in resposta.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if not bloco:
                        continue

                    arquivo.write(bloco)
                    barra.update(len(bloco))


def extrair_zip(
    arquivo_zip: Path,
    destino: Path,
) -> None:
    marcador = destino / ".extraido"

    if marcador.exists():
        print("Já extraído:", destino)
        return

    with zipfile.ZipFile(arquivo_zip) as arquivo:
        arquivo.extractall(destino)

    marcador.touch()


if CFG.usar_esc50:
    zip_esc = DADOS / "esc50_master.zip"
    baixar(URL_ESC50, zip_esc)
    extrair_zip(zip_esc, ESC50)

if CFG.usar_tut_rare:
    zip_tut = DADOS / "tut_eventos.zip"
    baixar(URL_TUT_EVENTOS, zip_tut)
    extrair_zip(zip_tut, TUT)

## 3. Montar metadados do ESC-50

In [ ]:
CLASSES_NORMAIS_ESC50 = {
    "rain",
    "sea_waves",
    "crickets",
    "chirping_birds",
    "water_drops",
    "wind",
    "footsteps",
    "laughing",
    "breathing",
    "snoring",
    "clock_tick",
    "washing_machine",
    "vacuum_cleaner",
    "engine",
    "train",
    "airplane",
    "helicopter",
    "fireworks",
    "thunderstorm",
    "siren",
    "clapping",
    "can_opening",
    "keyboard_typing",
    "mouse_click",
}

MAPA_ESC50 = {
    "crying_baby": "choro_distress",
    "glass_breaking": "vidro_quebrando",
    "door_wood_knock": "impacto",
}


def localizar_unico(
    raiz: Path,
    padrao: str,
) -> Path:
    encontrados = list(raiz.rglob(padrao))

    if not encontrados:
        raise FileNotFoundError(
            f"Não encontrei {padrao} em {raiz}"
        )

    return encontrados[0]


registros_esc = []

if CFG.usar_esc50:
    csv_esc = localizar_unico(
        ESC50,
        "esc50.csv",
    )
    audio_esc = localizar_unico(
        ESC50,
        "audio",
    )

    tabela_esc = pd.read_csv(csv_esc)

    for linha in tabela_esc.itertuples():
        categoria = str(linha.category)

        if categoria in MAPA_ESC50:
            classe = MAPA_ESC50[categoria]
        elif categoria in CLASSES_NORMAIS_ESC50:
            classe = "ambiente_normal"
        else:
            continue

        caminho = (
            audio_esc / str(linha.filename)
        ).resolve()

        if not caminho.exists():
            continue

        registros_esc.append(
            {
                "arquivo": str(caminho),
                "classe": classe,
                "fonte": "ESC-50",
                "grupo": (
                    f"esc50:{linha.src_file}"
                ),
                "origem_id": str(linha.src_file),
                "consulta": categoria,
                "licenca": "CC BY-NC",
            }
        )

df_esc = pd.DataFrame(registros_esc)

print("Registros ESC-50:", len(df_esc))

if not df_esc.empty:
    display(
        df_esc.groupby("classe")
        .size()
        .rename("quantidade")
        .to_frame()
    )

## 4. Montar metadados do TUT Rare

In [ ]:
EXTENSOES_AUDIO = {
    ".wav",
    ".flac",
    ".ogg",
    ".mp3",
    ".m4a",
}


def inferir_classe_tut(
    caminho: Path,
) -> str | None:
    texto = (
        str(caminho)
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )

    if any(
        termo in texto
        for termo in (
            "babycry",
            "baby_cry",
            "babycrying",
            "baby_crying",
        )
    ):
        return "choro_distress"

    if any(
        termo in texto
        for termo in (
            "glassbreak",
            "glass_break",
            "glassbreaking",
            "glass_breaking",
        )
    ):
        return "vidro_quebrando"

    if any(
        termo in texto
        for termo in (
            "gunshot",
            "gun_shot",
            "gunfire",
        )
    ):
        return "tiro"

    return None


registros_tut = []

if CFG.usar_tut_rare:
    arquivos_tut = [
        caminho
        for caminho in TUT.rglob("*")
        if (
            caminho.is_file()
            and caminho.suffix.lower()
            in EXTENSOES_AUDIO
        )
    ]

    for caminho in arquivos_tut:
        classe = inferir_classe_tut(caminho)

        if classe is None:
            continue

        registros_tut.append(
            {
                "arquivo": str(
                    caminho.resolve()
                ),
                "classe": classe,
                "fonte": "TUT Rare 2017",
                "grupo": (
                    f"tut:{caminho.stem}"
                ),
                "origem_id": caminho.stem,
                "consulta": classe,
                "licenca": (
                    "Conforme LICENSE.txt do TUT"
                ),
            }
        )

df_tut = pd.DataFrame(registros_tut)

print("Registros TUT:", len(df_tut))

if not df_tut.empty:
    display(
        df_tut.groupby("classe")
        .size()
        .rename("quantidade")
        .to_frame()
    )

if CFG.usar_tut_rare and df_tut.empty:
    print(
        "Exemplos de caminhos extraídos para diagnóstico:"
    )
    for caminho in list(TUT.rglob("*"))[:40]:
        print(caminho)
    raise RuntimeError(
        "Não foi possível identificar as classes do TUT."
    )

## 5. Baixar classes vocais e tiros pelo Kaggle

Esta versão não depende da API do Freesound.

Datasets usados:

- **ASVP-ESD**:
  - `04` → `choro_distress`
  - `05` → `voz_agressiva`
  - `06` → `grito_panico`
  - `02` → `ambiente_normal`
- **Gunshot Audio Dataset**:
  - todos os áudios → `tiro`

Observação importante:

- `grito_panico` representa o evento acústico;
- palavras como “socorro” pertencem ao ramo `VAD → Whisper → NLP`;
- `voz_agressiva` representa raiva, grunhido e frustração, não garante que exista uma discussão entre duas pessoas.

In [ ]:
SLUG_ASVP = (
    "dejolilandry/"
    "asvpesdspeech-nonspeech-emotional-utterances"
)
SLUG_TIRO = (
    "emrahaydemr/"
    "gunshot-audio-dataset"
)

PASTA_ASVP = Path(
    kagglehub.dataset_download(SLUG_ASVP)
)
PASTA_TIRO_KAGGLE = Path(
    kagglehub.dataset_download(SLUG_TIRO)
)

print("ASVP-ESD:", PASTA_ASVP)
print("Gunshot:", PASTA_TIRO_KAGGLE)

In [ ]:
MAPA_EMOCAO_ASVP = {
    "02": "ambiente_normal",
    "04": "choro_distress",
    "05": "voz_agressiva",
    "06": "grito_panico",
}


def codigo_emocao_asvp(
    caminho: Path,
) -> str | None:
    # A estrutura dos nomes de arquivo do ASVP-ESD é
    # Modality-Vocal_Channel-Emotion-Emotional_Intensity-Statement-Repetition-Actor
    # A emoção é o terceiro componente (índice 2).
    partes = caminho.stem.split("-")

    if len(partes) < 3:
        return None

    primeiro = partes[2].zfill(2)

    if primeiro in MAPA_EMOCAO_ASVP:
        return primeiro

    return None


registros_asvp = []

arquivos_asvp = [
    caminho
    for caminho in PASTA_ASVP.rglob("*")
    if (
        caminho.is_file()
        and caminho.suffix.lower()
        in EXTENSOES_AUDIO
    )
]

for caminho in arquivos_asvp:
    codigo = codigo_emocao_asvp(caminho)

    if codigo is None:
        continue

    classe = MAPA_EMOCAO_ASVP[codigo]

    # Mantém o mesmo ator no mesmo split.
    ator = caminho.parent.name

    registros_asvp.append(
        {
            "arquivo": str(caminho.resolve()),
            "classe": classe,
            "fonte": "ASVP-ESD Kaggle",
            "grupo": f"asvp:{ator}",
            "origem_id": caminho.stem,
            "consulta": f"emocao_{codigo}",
            "licenca": "Ver licença do dataset ASVP-ESD",
        }
    )

df_asvp = pd.DataFrame(registros_asvp)

print("Arquivos ASVP encontrados:", len(arquivos_asvp))
print("Registros ASVP utilizados:", len(df_asvp))

if df_asvp.empty:
    print("Primeiros nomes encontrados para diagnóstico:")

    for caminho in arquivos_asvp[:30]:
        print(caminho)

    raise RuntimeError(
        "Não foi possível interpretar os nomes do ASVP-ESD."
    )

display(
    df_asvp.groupby("classe")
    .size()
    .rename("quantidade")
    .to_frame()
)

In [ ]:
registros_tiro_kaggle = []

arquivos_tiro = [
    caminho
    for caminho in PASTA_TIRO_KAGGLE.rglob("*")
    if (
        caminho.is_file()
        and caminho.suffix.lower()
        in EXTENSOES_AUDIO
    )
]

for caminho in arquivos_tiro:
    registros_tiro_kaggle.append(
        {
            "arquivo": str(caminho.resolve()),
            "classe": "tiro",
            "fonte": "Gunshot Kaggle",
            "grupo": (
                "gunshot_kaggle:"
                f"{caminho.parent.name}:"
                f"{caminho.stem}"
            ),
            "origem_id": caminho.stem,
            "consulta": "gunshot",
            "licenca": "Ver licença do dataset no Kaggle",
        }
    )

df_tiro_kaggle = pd.DataFrame(
    registros_tiro_kaggle
)

print(
    "Registros de tiro Kaggle:",
    len(df_tiro_kaggle),
)

if df_tiro_kaggle.empty:
    raise RuntimeError(
        "Nenhum áudio foi encontrado no dataset de tiros."
    )

### Distribuição esperada após o Kaggle

O ASVP-ESD fornece centenas de exemplos de:

- choro;
- raiva, grunhido e frustração;
- grito, medo e pânico;
- voz neutra e calma.

O dataset de tiros adiciona centenas de recortes de disparos.

A auditoria manual continua necessária porque o ASVP-ESD reúne material realista de filmes, televisão, YouTube e outras fontes.

## 6. Unificar e limitar as fontes por classe

In [ ]:
tabelas = [
    tabela
    for tabela in (
        df_esc,
        df_tut,
        df_asvp,
        df_tiro_kaggle,
    )
    if not tabela.empty
]

metadados = pd.concat(
    tabelas,
    ignore_index=True,
)

metadados = metadados[
    metadados["classe"].isin(CLASSES)
].copy()

metadados = metadados.drop_duplicates(
    subset=["grupo", "classe"]
)

partes_limitadas = []

for classe in CLASSES:
    parte = metadados[
        metadados["classe"] == classe
    ].copy()

    grupos = parte["grupo"].drop_duplicates().tolist()
    random.Random(CFG.semente).shuffle(grupos)

    grupos = grupos[
        : CFG.max_fontes_por_classe
    ]

    partes_limitadas.append(
        parte[parte["grupo"].isin(grupos)]
    )

metadados = pd.concat(
    partes_limitadas,
    ignore_index=True,
)

metadados.to_csv(
    RESULTADOS / "fontes_unificadas.csv",
    index=False,
)

resumo_fontes = (
    metadados.groupby(["classe", "fonte"])
    .size()
    .unstack(fill_value=0)
)

display(resumo_fontes)

ausentes = [
    classe
    for classe in CLASSES
    if classe not in set(
        metadados["classe"]
    )
]

if ausentes:
    raise RuntimeError(
        f"Classes sem dados: {ausentes}"
    )

## 7. Auditoria manual

Ouça exemplos aleatórios. A coleta do Freesound usa metadados e busca textual, então alguns arquivos podem estar rotulados incorretamente.

Remova linhas ruins de `fontes_unificadas.csv` ou marque `APROVAR_AUDITORIA = True` somente depois de conferir.

In [ ]:
AMOSTRAS_POR_CLASSE = 3

for classe in CLASSES:
    print("\n" + "=" * 72)
    print("Classe:", classe)

    parte = metadados[
        metadados["classe"] == classe
    ]

    quantidade = min(
        AMOSTRAS_POR_CLASSE,
        len(parte),
    )

    amostras = parte.sample(
        quantidade,
        random_state=CFG.semente,
    )

    for linha in amostras.itertuples():
        print(
            linha.fonte,
            "|",
            Path(linha.arquivo).name,
            "|",
            getattr(linha, "consulta", ""),
        )
        display(Audio(linha.arquivo))

In [ ]:
APROVAR_AUDITORIA = True

if not APROVAR_AUDITORIA:
    raise RuntimeError(
        "Ouça as amostras e altere "
        "APROVAR_AUDITORIA para True."
    )

## 8. Separação por grupo antes do augmentation

O split é feito por `grupo` dentro de cada classe. Assim, o mesmo áudio original e suas variações nunca atravessam treino, validação e teste.

In [ ]:
def separar_grupos_globais(
    grupos_unicos: list[str],
    semente: int,
) -> dict[str, set[str]]:
    random.Random(semente).shuffle(grupos_unicos)

    n = len(grupos_unicos)

    if n < 3:
        raise RuntimeError(
            "São necessários pelo menos 3 grupos únicos "
            "no total para realizar a separação de splits."
        )

    # Calculate target sizes based on proportions, ensuring minimum 1 for each main split
    n_treino = max(1, int(round(n * CFG.proporcao_treino)))
    n_validacao = max(1, int(round(n * CFG.proporcao_validacao)))

    # Adjust if train and validation sizes combined exceed total available groups for test split
    if n_treino + n_validacao >= n:
        # If there are enough groups (n >= 3), ensure each split gets at least one
        n_treino = max(1, n - 2) # Allocate 1 for validation and 1 for test
        n_validacao = 1

    # Perform the split
    treino = grupos_unicos[:n_treino]
    validacao = grupos_unicos[n_treino : n_treino + n_validacao]
    teste = grupos_unicos[n_treino + n_validacao :]

    # Robustness checks for empty splits after initial assignment due to rounding or small n
    if not teste:
        if validacao:
            teste.append(validacao.pop()) # Move one from valid to test
        elif treino:
            teste.append(treino.pop()) # Move one from train to test
        else: # Should not happen if n >= 3
            raise RuntimeError("Não foi possível criar um split de teste adequado.")

    if not validacao:
        if treino:
            validacao.append(treino.pop()) # Move one from train to valid
        else: # Should not happen if n >= 3
            raise RuntimeError("Não foi possível criar um split de validação adequado.")

    if not treino: # Should not happen if n >= 3
        raise RuntimeError("Não foi possível criar um split de treino adequado.")

    return {
        "treino": set(treino),
        "validacao": set(validacao),
        "teste": set(teste),
    }


# --- Main execution part of the cell ---

# Get all unique groups from the entire dataset
todos_grupos_unicos = metadados["grupo"].unique().tolist()

# Perform the global split of these unique groups
divisao_global = separar_grupos_globais(
    todos_grupos_unicos,
    CFG.semente, # Use a fixed seed for reproducibility of the global split
)

# Create a mapping from each group to its assigned split
mapa_grupo_para_split = {}
for split_name, grupos_set in divisao_global.items():
    for grupo in grupos_set:
        mapa_grupo_para_split[grupo] = split_name

# Apply this mapping to the metadados DataFrame to assign the 'split' column
metadados["split"] = metadados["grupo"].map(mapa_grupo_para_split)

# Check for classes that might be completely missing from any split after global splitting
# This is crucial because a group might contain only one class, and if that group ends up
# in 'test', that class might be missing from 'train' or 'validation'.
for classe in CLASSES:
    for split_name in ["treino", "validacao", "teste"]:
        if metadados[(metadados["classe"] == classe) & (metadados["split"] == split_name)].empty:
            raise RuntimeError(
                f"A classe '{classe}' não possui amostras no split de '{split_name}' "
                "após a divisão global dos grupos. Considere aumentar o número "
                "de grupos ou ajustar as proporções de split."
            )

metadados.to_csv(
    RESULTADOS / "fontes_com_split.csv",
    index=False,
)

tabela_split = (
    metadados.groupby(["classe", "split"])
    ["grupo"]
    .nunique()
    .unstack(fill_value=0)
)

display(tabela_split)

vazamento = (
    metadados.groupby("grupo")["split"]
    .nunique()
    .gt(1)
    .sum()
)

print("Grupos em mais de um split:", vazamento)

if vazamento:
    raise RuntimeError(
        "Foi detectado vazamento entre splits."
    )

## 9. Janelas, augmentation, log-mel e MFCC

- Janela: 2 segundos;
- passo de treino: 1 segundo;
- eventos: seleciona as janelas com maior energia;
- ambiente normal: seleciona janelas distribuídas;
- augmentation: somente no treino;
- entradas do modelo: log-mel e MFCC.

In [ ]:
AMOSTRAS_JANELA = int(
    CFG.taxa_amostragem
    * CFG.duracao_janela
)

exemplo_zero = np.zeros(
    AMOSTRAS_JANELA,
    dtype=np.float32,
)

mel_zero = librosa.feature.melspectrogram(
    y=exemplo_zero,
    sr=CFG.taxa_amostragem,
    n_fft=CFG.n_fft,
    hop_length=CFG.hop_length,
    n_mels=CFG.n_mels,
    fmin=CFG.fmin,
    fmax=CFG.fmax,
    power=2.0,
)

FRAMES = mel_zero.shape[1]

print(
    "Entrada log-mel:",
    (CFG.n_mels, FRAMES, 1),
)
print(
    "Entrada MFCC:",
    (CFG.n_mfcc, FRAMES, 1),
)


def carregar_audio(
    caminho: str | Path,
) -> np.ndarray:
    audio, _ = librosa.load(
        str(caminho),
        sr=CFG.taxa_amostragem,
        mono=True,
    )

    audio = np.nan_to_num(
        audio.astype(np.float32)
    )

    pico = np.max(np.abs(audio))

    if pico > 1.0:
        audio = audio / pico

    return audio


def ajustar_tamanho(
    audio: np.ndarray,
) -> np.ndarray:
    if len(audio) < AMOSTRAS_JANELA:
        return np.pad(
            audio,
            (
                0,
                AMOSTRAS_JANELA
                - len(audio),
            ),
        )

    return audio[:AMOSTRAS_JANELA]


def gerar_janelas_audio(
    audio: np.ndarray,
    classe: str,
) -> list[np.ndarray]:
    if len(audio) <= AMOSTRAS_JANELA:
        return [
            ajustar_tamanho(audio)
        ]

    passo = int(
        CFG.taxa_amostragem
        * CFG.passo_janela_treino
    )

    inicios = list(
        range(
            0,
            len(audio)
            - AMOSTRAS_JANELA
            + 1,
            passo,
        )
    )

    if not inicios:
        inicios = [0]

    candidatos = [
        audio[
            inicio :
            inicio + AMOSTRAS_JANELA
        ]
        for inicio in inicios
    ]

    limite = CFG.max_janelas_por_fonte

    if len(candidatos) <= limite:
        return [
            ajustar_tamanho(janela)
            for janela in candidatos
        ]

    if classe == "ambiente_normal":
        indices = np.linspace(
            0,
            len(candidatos) - 1,
            limite,
            dtype=int,
        )
    else:
        energias = [
            float(
                np.sqrt(
                    np.mean(
                        np.square(janela)
                    )
                    + 1e-10
                )
            )
            for janela in candidatos
        ]
        indices = np.argsort(
            energias
        )[-limite:]

    return [
        ajustar_tamanho(
            candidatos[int(indice)]
        )
        for indice in sorted(indices)
    ]


def augmentar(
    audio: np.ndarray,
    rng: np.random.Generator,
) -> np.ndarray:
    saida = audio.copy()

    ganho_db = rng.uniform(-8.0, 6.0)
    saida *= 10 ** (ganho_db / 20.0)

    deslocamento = int(
        rng.uniform(-0.15, 0.15)
        * len(saida)
    )
    saida = np.roll(
        saida,
        deslocamento,
    )

    if rng.random() < 0.50:
        snr_db = rng.uniform(10.0, 30.0)
        potencia_sinal = (
            np.mean(saida ** 2) + 1e-10
        )
        potencia_ruido = (
            potencia_sinal
            / (10 ** (snr_db / 10.0))
        )
        ruido = rng.normal(
            0.0,
            np.sqrt(potencia_ruido),
            size=saida.shape,
        )
        saida = saida + ruido

    pico = np.max(np.abs(saida))

    if pico > 1.0:
        saida = saida / pico

    return saida.astype(np.float32)


def normalizar_matriz(
    matriz: np.ndarray,
) -> np.ndarray:
    media = float(np.mean(matriz))
    desvio = float(np.std(matriz))

    return (
        (matriz - media)
        / (desvio + 1e-6)
    ).astype(np.float32)


def extrair_features(
    audio: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    audio = ajustar_tamanho(audio)

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=CFG.taxa_amostragem,
        n_fft=CFG.n_fft,
        hop_length=CFG.hop_length,
        n_mels=CFG.n_mels,
        fmin=CFG.fmin,
        fmax=CFG.fmax,
        power=2.0,
    )

    log_mel = librosa.power_to_db(
        mel,
        ref=np.max,
        top_db=80.0,
    )

    mfcc = librosa.feature.mfcc(
        S=log_mel,
        n_mfcc=CFG.n_mfcc,
    )

    log_mel = normalizar_matriz(
        log_mel
    )
    mfcc = normalizar_matriz(mfcc)

    return (
        log_mel[..., np.newaxis],
        mfcc[..., np.newaxis],
    )

In [ ]:
def processar_split(
    nome_split: str,
) -> tuple[
    np.ndarray,
    np.ndarray,
    np.ndarray,
    pd.DataFrame,
]:
    parte = metadados[
        metadados["split"] == nome_split
    ].copy()

    mel_lista = []
    mfcc_lista = []
    rotulos = []
    registros = []

    rng = np.random.default_rng(
        CFG.semente
        + {
            "treino": 1,
            "validacao": 2,
            "teste": 3,
        }[nome_split]
    )

    for linha in tqdm(
        parte.itertuples(),
        total=len(parte),
        desc=f"Processando {nome_split}",
    ):
        try:
            audio = carregar_audio(
                linha.arquivo
            )
            janelas = gerar_janelas_audio(
                audio,
                linha.classe,
            )
        except Exception:
            continue

        for indice_janela, janela in enumerate(janelas):
            versoes = [("original", janela)]

            if nome_split == "treino":
                for copia in range(
                    CFG.copias_augmentation_treino
                ):
                    versoes.append(
                        (
                            f"aug_{copia + 1}",
                            augmentar(
                                janela,
                                rng,
                            ),
                        )
                    )

            for versao, audio_versao in versoes:
                log_mel, mfcc = extrair_features(
                    audio_versao
                )

                mel_lista.append(log_mel)
                mfcc_lista.append(mfcc)
                rotulos.append(
                    ID_CLASSE[linha.classe]
                )
                registros.append(
                    {
                        "split": nome_split,
                        "classe": linha.classe,
                        "grupo": linha.grupo,
                        "arquivo": linha.arquivo,
                        "janela": indice_janela,
                        "versao": versao,
                    }
                )

    return (
        np.asarray(
            mel_lista,
            dtype=np.float32,
        ),
        np.asarray(
            mfcc_lista,
            dtype=np.float32,
        ),
        np.asarray(
            rotulos,
            dtype=np.int64,
        ),
        pd.DataFrame(registros),
    )


dados_processados = {}

for split in (
    "treino",
    "validacao",
    "teste",
):
    cache = PROCESSADO / f"{split}.npz"
    cache_meta = (
        PROCESSADO / f"{split}.csv"
    )

    if cache.exists() and cache_meta.exists():
        arquivo = np.load(cache)
        dados_processados[split] = (
            arquivo["mel"],
            arquivo["mfcc"],
            arquivo["y"],
            pd.read_csv(cache_meta),
        )
        print("Cache carregado:", split)
        continue

    mel, mfcc, y, meta = processar_split(
        split
    )

    np.savez_compressed(
        cache,
        mel=mel,
        mfcc=mfcc,
        y=y,
    )
    meta.to_csv(
        cache_meta,
        index=False,
    )

    dados_processados[split] = (
        mel,
        mfcc,
        y,
        meta,
    )

for split, (
    mel,
    mfcc,
    y,
    meta,
) in dados_processados.items():
    print(
        split,
        "mel=",
        mel.shape,
        "mfcc=",
        mfcc.shape,
        "y=",
        y.shape,
    )

## 10. Visualizar log-mel e MFCC

In [ ]:
X_mel_treino, X_mfcc_treino, y_treino, meta_treino = (
    dados_processados["treino"]
)
X_mel_val, X_mfcc_val, y_val, meta_val = (
    dados_processados["validacao"]
)
X_mel_teste, X_mfcc_teste, y_teste, meta_teste = (
    dados_processados["teste"]
)

indice = 0

plt.figure(figsize=(12, 4))
plt.imshow(
    X_mel_treino[indice, :, :, 0],
    aspect="auto",
    origin="lower",
)
plt.title(
    "Log-mel — "
    + CLASSES[int(y_treino[indice])]
)
plt.xlabel("Tempo")
plt.ylabel("Bandas mel")
plt.colorbar()
plt.show()

plt.figure(figsize=(12, 4))
plt.imshow(
    X_mfcc_treino[indice, :, :, 0],
    aspect="auto",
    origin="lower",
)
plt.title(
    "MFCC — "
    + CLASSES[int(y_treino[indice])]
)
plt.xlabel("Tempo")
plt.ylabel("Coeficientes MFCC")
plt.colorbar()
plt.show()

## 11. Criar a CNN leve de duas entradas

In [ ]:
def ramo_convolucional(
    entrada: tf.Tensor,
    filtros_iniciais: int,
) -> tf.Tensor:
    x = tf.keras.layers.Conv2D(
        filtros_iniciais,
        3,
        padding="same",
        use_bias=False,
    )(entrada)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)

    x = tf.keras.layers.SeparableConv2D(
        filtros_iniciais * 2,
        3,
        padding="same",
        use_bias=False,
    )(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)

    x = tf.keras.layers.SeparableConv2D(
        filtros_iniciais * 4,
        3,
        padding="same",
        use_bias=False,
    )(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    return x


entrada_mel = tf.keras.Input(
    shape=(CFG.n_mels, FRAMES, 1),
    name="log_mel",
)
entrada_mfcc = tf.keras.Input(
    shape=(CFG.n_mfcc, FRAMES, 1),
    name="mfcc",
)

ramo_mel = ramo_convolucional(
    entrada_mel,
    16,
)
ramo_mfcc = ramo_convolucional(
    entrada_mfcc,
    8,
)

x = tf.keras.layers.Concatenate()(
    [ramo_mel, ramo_mfcc]
)
x = tf.keras.layers.Dense(
    64,
    activation="relu",
)(x)
x = tf.keras.layers.Dropout(0.35)(x)

saida = tf.keras.layers.Dense(
    len(CLASSES),
    activation="softmax",
    name="probabilidades",
)(x)

modelo = tf.keras.Model(
    inputs={
        "log_mel": entrada_mel,
        "mfcc": entrada_mfcc,
    },
    outputs=saida,
    name="aed_multiclasse_mel_mfcc",
)

modelo.compile(
    optimizer=tf.keras.optimizers.Adam(
        CFG.taxa_aprendizado
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

modelo.summary()

## 12. Treinar

In [ ]:
pesos = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(CLASSES)),
    y=y_treino,
)

pesos_classes = {
    indice: float(peso)
    for indice, peso in enumerate(pesos)
}

print("Pesos:", pesos_classes)

caminho_melhor = (
    MODELOS / "aed_melhor.keras"
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        caminho_melhor,
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=CFG.paciencia,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(
        RESULTADOS
        / "historico_treinamento.csv"
    ),
]

historico = modelo.fit(
    {
        "log_mel": X_mel_treino,
        "mfcc": X_mfcc_treino,
    },
    y_treino,
    validation_data=(
        {
            "log_mel": X_mel_val,
            "mfcc": X_mfcc_val,
        },
        y_val,
    ),
    epochs=CFG.epocas,
    batch_size=CFG.tamanho_lote,
    class_weight=pesos_classes,
    callbacks=callbacks,
    verbose=1,
)

modelo = tf.keras.models.load_model(
    caminho_melhor
)

In [ ]:
historico_df = pd.DataFrame(
    historico.history
)

plt.figure(figsize=(10, 5))
plt.plot(
    historico_df["loss"],
    label="treino",
)
plt.plot(
    historico_df["val_loss"],
    label="validação",
)
plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("Loss")
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(
    historico_df["accuracy"],
    label="treino",
)
plt.plot(
    historico_df["val_accuracy"],
    label="validação",
)
plt.xlabel("Época")
plt.ylabel("Acurácia")
plt.title("Acurácia")
plt.legend()
plt.grid()
plt.show()

## 13. Avaliar no teste

In [ ]:
probabilidades_teste = modelo.predict(
    {
        "log_mel": X_mel_teste,
        "mfcc": X_mfcc_teste,
    },
    batch_size=CFG.tamanho_lote,
    verbose=1,
)

previsoes_teste = np.argmax(
    probabilidades_teste,
    axis=1,
)

relatorio_texto = classification_report(
    y_teste,
    previsoes_teste,
    target_names=CLASSES,
    digits=4,
)

print(relatorio_texto)

relatorio_dict = classification_report(
    y_teste,
    previsoes_teste,
    target_names=CLASSES,
    output_dict=True,
)

pd.DataFrame(relatorio_dict).T.to_csv(
    RESULTADOS
    / "metricas_teste_por_classe.csv"
)

matriz = confusion_matrix(
    y_teste,
    previsoes_teste,
)

plt.figure(figsize=(10, 8))
sns.heatmap(
    matriz,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASSES,
    yticklabels=CLASSES,
)
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.title("Matriz de confusão — teste")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(
    RESULTADOS
    / "matriz_confusao_teste.png",
    dpi=160,
)
plt.show()

predicoes_df = meta_teste.copy()
predicoes_df["classe_real"] = [
    CLASSES[int(indice)]
    for indice in y_teste
]
predicoes_df["classe_prevista"] = [
    CLASSES[int(indice)]
    for indice in previsoes_teste
]
predicoes_df["confianca"] = np.max(
    probabilidades_teste,
    axis=1,
)
predicoes_df.to_csv(
    RESULTADOS
    / "predicoes_teste.csv",
    index=False,
)

## 14. Calibrar limiares por classe

Para cada evento, o notebook escolhe no conjunto de validação o limiar que maximiza o F1 um-contra-todos.

Esses limiares são usados na inferência contínua.

In [ ]:
probabilidades_val = modelo.predict(
    {
        "log_mel": X_mel_val,
        "mfcc": X_mfcc_val,
    },
    batch_size=CFG.tamanho_lote,
    verbose=0,
)

limiares = {}
registros_limiares = []

for indice, classe in enumerate(CLASSES):
    y_binario = (
        y_val == indice
    ).astype(int)

    melhor_limiar = 0.50
    melhor_f1 = -1.0

    for limiar in np.arange(
        0.30,
        0.91,
        0.02,
    ):
        previsto = (
            probabilidades_val[:, indice]
            >= limiar
        ).astype(int)

        valor_f1 = f1_score(
            y_binario,
            previsto,
            zero_division=0,
        )

        if valor_f1 > melhor_f1:
            melhor_f1 = valor_f1
            melhor_limiar = float(
                round(limiar, 2)
            )

    limiares[classe] = melhor_limiar
    registros_limiares.append(
        {
            "classe": classe,
            "limiar": melhor_limiar,
            "f1_validacao": melhor_f1,
        }
    )

tabela_limiares = pd.DataFrame(
    registros_limiares
)

display(tabela_limiares)

tabela_limiares.to_csv(
    RESULTADOS
    / "limiares_por_classe.csv",
    index=False,
)

(
    MODELOS / "limiares.json"
).write_text(
    json.dumps(
        limiares,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

## 15. Exportar Keras e TFLite

In [ ]:
caminho_final_keras = (
    MODELOS / "aed_multiclasse_final.keras"
)
modelo.save(caminho_final_keras)

converter = tf.lite.TFLiteConverter.from_keras_model(
    modelo
)
tflite_float32 = converter.convert()

caminho_float32 = (
    MODELOS / "aed_float32.tflite"
)
caminho_float32.write_bytes(
    tflite_float32
)

converter = tf.lite.TFLiteConverter.from_keras_model(
    modelo
)
converter.optimizations = [
    tf.lite.Optimize.DEFAULT
]
converter.target_spec.supported_types = [
    tf.float16
]
tflite_float16 = converter.convert()

caminho_float16 = (
    MODELOS / "aed_float16.tflite"
)
caminho_float16.write_bytes(
    tflite_float16
)


def dados_representativos():
    limite = min(
        300,
        len(X_mel_treino),
    )

    for indice in range(limite):
        yield [
            X_mel_treino[
                indice : indice + 1
            ],
            X_mfcc_treino[
                indice : indice + 1
            ],
        ]


caminho_int8 = (
    MODELOS / "aed_int8.tflite"
)

try:
    converter = (
        tf.lite.TFLiteConverter
        .from_keras_model(modelo)
    )
    converter.optimizations = [
        tf.lite.Optimize.DEFAULT
    ]
    converter.representative_dataset = (
        dados_representativos
    )
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet
        .TFLITE_BUILTINS_INT8
    ]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    tflite_int8 = converter.convert()
    caminho_int8.write_bytes(
        tflite_int8
    )
    print("INT8 exportado.")
except Exception as erro:
    print(
        "Exportação INT8 falhou, "
        "mas float32/float16 foram salvos:",
        erro,
    )

for caminho in (
    caminho_final_keras,
    caminho_float32,
    caminho_float16,
    caminho_int8,
):
    if caminho.exists():
        print(
            caminho.name,
            f"{caminho.stat().st_size / 1e6:.2f} MB",
        )

## 16. Conferir paridade Keras × TFLite

In [ ]:
def prever_tflite_float(
    caminho: Path,
    mel: np.ndarray,
    mfcc: np.ndarray,
) -> np.ndarray:
    interpretador = tf.lite.Interpreter(
        model_path=str(caminho)
    )
    interpretador.allocate_tensors()

    entradas = (
        interpretador.get_input_details()
    )
    saida = (
        interpretador.get_output_details()[0]
    )

    mapa = {
        detalhe["name"]: detalhe
        for detalhe in entradas
    }

    for detalhe in entradas:
        nome = detalhe["name"].lower()

        if "log_mel" in nome:
            valor = mel.astype(
                detalhe["dtype"]
            )
        elif "mfcc" in nome:
            valor = mfcc.astype(
                detalhe["dtype"]
            )
        else:
            # Fallback por formato.
            if detalhe["shape"][1] == CFG.n_mels:
                valor = mel.astype(
                    detalhe["dtype"]
                )
            else:
                valor = mfcc.astype(
                    detalhe["dtype"]
                )

        interpretador.set_tensor(
            detalhe["index"],
            valor,
        )

    interpretador.invoke()

    return interpretador.get_tensor(
        saida["index"]
    )


indice = 0

pred_keras = modelo.predict(
    {
        "log_mel": X_mel_teste[
            indice : indice + 1
        ],
        "mfcc": X_mfcc_teste[
            indice : indice + 1
        ],
    },
    verbose=0,
)

pred_float32 = prever_tflite_float(
    caminho_float32,
    X_mel_teste[indice : indice + 1],
    X_mfcc_teste[indice : indice + 1],
)

pred_float16 = prever_tflite_float(
    caminho_float16,
    X_mel_teste[indice : indice + 1],
    X_mfcc_teste[indice : indice + 1],
)

print(
    "Máxima diferença float32:",
    float(
        np.max(
            np.abs(
                pred_keras
                - pred_float32
            )
        )
    ),
)
print(
    "Máxima diferença float16:",
    float(
        np.max(
            np.abs(
                pred_keras
                - pred_float16
            )
        )
    ),
)

## 17. Inferência contínua em arquivo externo

A decisão temporal exige o mesmo evento perigoso em pelo menos duas janelas consecutivas.

A classe `ambiente_normal` nunca gera alerta.

In [ ]:
from google.colab import files

enviados = files.upload()

arquivos_audio = [
    nome
    for nome in enviados
    if Path(nome).suffix.lower()
    in EXTENSOES_AUDIO
]

if len(arquivos_audio) != 1:
    raise RuntimeError(
        "Envie exatamente um arquivo de áudio."
    )

ARQUIVO_TESTE = Path(
    arquivos_audio[0]
).resolve()

print("Áudio externo:", ARQUIVO_TESTE)
display(Audio(str(ARQUIVO_TESTE)))

In [ ]:
def janelas_deslizantes(
    audio: np.ndarray,
    passo_segundos: float,
) -> list[tuple[float, np.ndarray]]:
    passo = int(
        CFG.taxa_amostragem
        * passo_segundos
    )

    if len(audio) <= AMOSTRAS_JANELA:
        return [
            (
                0.0,
                ajustar_tamanho(audio),
            )
        ]

    resultado = []

    for inicio in range(
        0,
        len(audio)
        - AMOSTRAS_JANELA
        + 1,
        passo,
    ):
        resultado.append(
            (
                inicio
                / CFG.taxa_amostragem,
                audio[
                    inicio :
                    inicio + AMOSTRAS_JANELA
                ],
            )
        )

    return resultado


def inferir_audio_continuo(
    caminho: Path,
) -> pd.DataFrame:
    audio = carregar_audio(caminho)
    janelas = janelas_deslizantes(
        audio,
        CFG.passo_janela_inferencia,
    )

    mel_lista = []
    mfcc_lista = []
    inicios = []

    for inicio, janela in janelas:
        log_mel, mfcc = extrair_features(
            janela
        )
        mel_lista.append(log_mel)
        mfcc_lista.append(mfcc)
        inicios.append(inicio)

    probs = modelo.predict(
        {
            "log_mel": np.asarray(
                mel_lista,
                dtype=np.float32,
            ),
            "mfcc": np.asarray(
                mfcc_lista,
                dtype=np.float32,
            ),
        },
        batch_size=CFG.tamanho_lote,
        verbose=0,
    )

    linhas = []

    for inicio, vetor in zip(
        inicios,
        probs,
        strict=True,
    ):
        indice = int(np.argmax(vetor))
        classe = CLASSES[indice]
        confianca = float(vetor[indice])
        limiar = float(
            limiares[classe]
        )

        ativo = (
            classe != "ambiente_normal"
            and confianca >= limiar
        )

        linhas.append(
            {
                "inicio_segundos": inicio,
                "fim_segundos": (
                    inicio
                    + CFG.duracao_janela
                ),
                "classe": classe,
                "confianca": confianca,
                "limiar": limiar,
                "evento_ativo": ativo,
            }
        )

    df = pd.DataFrame(linhas)

    consecutivas = 0
    classe_anterior = None
    alertas = []

    for linha in df.itertuples():
        if (
            linha.evento_ativo
            and linha.classe
            == classe_anterior
        ):
            consecutivas += 1
        elif linha.evento_ativo:
            consecutivas = 1
        else:
            consecutivas = 0

        classe_anterior = (
            linha.classe
            if linha.evento_ativo
            else None
        )

        alertas.append(
            consecutivas
            >= CFG.minimo_eventos_consecutivos
        )

    df["alerta_temporal"] = alertas
    return df


timeline = inferir_audio_continuo(
    ARQUIVO_TESTE
)

display(
    timeline.sort_values(
        "confianca",
        ascending=False,
    ).head(20)
)

caminho_timeline = (
    RESULTADOS
    / "timeline_audio_externo.csv"
)
timeline.to_csv(
    caminho_timeline,
    index=False,
)

print(
    "Alertas temporais:",
    int(
        timeline[
            "alerta_temporal"
        ].sum()
    ),
)

plt.figure(figsize=(14, 5))

for classe in CLASSES:
    parte = timeline[
        timeline["classe"] == classe
    ]

    if parte.empty:
        continue

    plt.scatter(
        parte["inicio_segundos"],
        parte["confianca"],
        label=classe,
        s=25,
    )

plt.xlabel("Tempo (s)")
plt.ylabel("Confiança")
plt.title("Linha do tempo do AED")
plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)
plt.grid()
plt.tight_layout()
plt.show()

## 18. VAD em paralelo

O VAD identifica trechos com voz para o ramo:

```text
VAD → Whisper → NLP
```

O resultado do VAD é apenas informativo aqui. O AED acima continua processando todas as janelas.

In [ ]:
def analisar_vad(
    caminho: Path,
    agressividade: int = 2,
    frame_ms: int = 30,
) -> dict[str, Any]:
    audio = carregar_audio(caminho)
    audio = np.clip(
        audio,
        -1.0,
        1.0,
    )

    pcm = (
        audio * 32767
    ).astype(np.int16)

    amostras_frame = int(
        CFG.taxa_amostragem
        * frame_ms
        / 1000
    )

    vad = webrtcvad.Vad(
        agressividade
    )

    decisoes = []
    tempos = []

    for inicio in range(
        0,
        len(pcm)
        - amostras_frame
        + 1,
        amostras_frame,
    ):
        frame = pcm[
            inicio :
            inicio + amostras_frame
        ]

        fala = vad.is_speech(
            frame.tobytes(),
            CFG.taxa_amostragem,
        )

        decisoes.append(fala)
        tempos.append(
            inicio
            / CFG.taxa_amostragem
        )

    proporcao = (
        float(np.mean(decisoes))
        if decisoes
        else 0.0
    )

    return {
        "frames": len(decisoes),
        "frames_fala": int(
            sum(decisoes)
        ),
        "proporcao_fala": proporcao,
        "tempos": tempos,
        "decisoes": decisoes,
    }


resultado_vad = analisar_vad(
    ARQUIVO_TESTE
)

print(
    "Frames analisados:",
    resultado_vad["frames"],
)
print(
    "Frames com fala:",
    resultado_vad["frames_fala"],
)
print(
    "Proporção de fala:",
    f"{resultado_vad['proporcao_fala']:.2%}",
)

print(
    "\nInterpretação:"
    "\n- AED: analisou todo o áudio."
    "\n- VAD: indica se o Whisper deve ser acionado."
)

## 19. Gerar configuração e pacote final

In [ ]:
configuracao_exportada = {
    "taxa_amostragem": CFG.taxa_amostragem,
    "duracao_janela": CFG.duracao_janela,
    "passo_janela_inferencia": (
        CFG.passo_janela_inferencia
    ),
    "n_fft": CFG.n_fft,
    "hop_length": CFG.hop_length,
    "n_mels": CFG.n_mels,
    "n_mfcc": CFG.n_mfcc,
    "fmin": CFG.fmin,
    "fmax": CFG.fmax,
    "frames": FRAMES,
    "classes": CLASSES,
    "limiares": limiares,
    "minimo_eventos_consecutivos": (
        CFG.minimo_eventos_consecutivos
    ),
    "vad": {
        "uso": (
            "Somente para acionar "
            "Whisper/NLP"
        ),
        "bloqueia_aed": False,
        "frame_ms": 30,
        "agressividade": 2,
    },
}

(
    MODELOS / "configuracao_aed.json"
).write_text(
    json.dumps(
        configuracao_exportada,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

if ENTREGA.exists():
    shutil.rmtree(ENTREGA)

ENTREGA.mkdir(parents=True)

for caminho in MODELOS.iterdir():
    if caminho.is_file():
        shutil.copy2(
            caminho,
            ENTREGA / caminho.name,
        )

for nome in (
    "historico_treinamento.csv",
    "metricas_teste_por_classe.csv",
    "predicoes_teste.csv",
    "matriz_confusao_teste.png",
    "limiares_por_classe.csv",
    "fontes_com_split.csv",
    "metadados_freesound.csv",
    "timeline_audio_externo.csv",
):
    origem = RESULTADOS / nome

    if origem.exists():
        shutil.copy2(
            origem,
            ENTREGA / nome,
        )

resumo = {
    "configuracao": asdict(CFG),
    "classes": CLASSES,
    "limiares": limiares,
    "quantidade_fontes": (
        metadados.groupby(
            ["classe", "split"]
        )["grupo"]
        .nunique()
        .reset_index()
        .to_dict(orient="records")
    ),
    "metricas_teste": relatorio_dict,
}

(
    ENTREGA
    / "resumo_experimento.json"
).write_text(
    json.dumps(
        resumo,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

arquivo_zip = shutil.make_archive(
    "/content/aed_multiclasse_entrega",
    "zip",
    root_dir=str(ENTREGA),
)

print("Pacote final:", arquivo_zip)

from google.colab import files
files.download(arquivo_zip)

## Próximas etapas no projeto

Depois de baixar o ZIP:

```text
models/audio/
├── aed_multiclasse_final.keras
├── aed_float32.tflite
├── aed_float16.tflite
├── aed_int8.tflite
├── limiares.json
└── configuracao_aed.json
```

Integração futura:

```text
INMP441
├── AED contínuo
├── VAD → faster-whisper → NLP
└── GCC-PHAT → TDOA → DOA
```

GCC-PHAT, TDOA e DOA não são treinados neste notebook; eles dependem da captura sincronizada dos microfones na Raspberry Pi.